# SemGAN inference demo

This notebook randomly selects ten test images, runs one SemGAN generator, 
saves the generated images, and displays every input/output pair side by side 
in a single comparison figure. Change `DIRECTION` to switch domains.

In [ ]:
from pathlib import Path
import random
import sys

import torch
from matplotlib import pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CONFIG = REPO_ROOT / "configs/semgan.yaml"
# Epoch-200 reference run; separate from the final proportional 5k checkpoint.
CHECKPOINT = REPO_ROOT / "checkpoints/lastchance_semgan_20260613_023510_latest.pt"
DIRECTION = "day2night"  # or "night2day"
INPUT_DIR = REPO_ROOT / ("data/testA" if DIRECTION == "day2night" else "data/testB")
OUTPUT_DIR = REPO_ROOT / "results/inference_demo"
NUM_SAMPLES = 10
SEED = 42

assert CONFIG.is_file(), CONFIG
assert CHECKPOINT.is_file(), CHECKPOINT
assert INPUT_DIR.is_dir(), INPUT_DIR
print(f"repository: {REPO_ROOT}")
print(f"input:      {INPUT_DIR}")
print(f"direction:  {DIRECTION}")


In [ ]:
from src.apply_cyclegan import (
    build_generator,
    build_inference_transform,
    find_checkpoint,
    gather_image_paths,
    tensor_to_pil,
)
from src.utils.common import load_cfg

cfg = load_cfg(CONFIG)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = find_checkpoint(str(CHECKPOINT), cfg)
extensions = cfg["data"].get("extensions", ("jpg", "jpeg", "png"))
image_paths = gather_image_paths(INPUT_DIR, extensions)
assert image_paths, f"No images found in {INPUT_DIR}."

sample_count = min(NUM_SAMPLES, len(image_paths))
sample_paths = random.Random(SEED).sample(image_paths, sample_count)
transform = build_inference_transform(cfg)
generator = build_generator(cfg, device, DIRECTION)
state = torch.load(checkpoint, map_location="cpu", weights_only=True)
state_key = "G" if DIRECTION == "day2night" else "F"
generator.load_state_dict(state[state_key])
generator.eval()

inference_cfg = cfg.get("inference", {})
brightness_gain = float(inference_cfg.get("brightness_gain", 1.0))
output_gamma = float(inference_cfg.get("output_gamma", 1.0))
print(f"checkpoint: {checkpoint}")
print(f"device:     {device}")
print(f"samples:    {sample_count}")


In [ ]:
result_dir = OUTPUT_DIR / DIRECTION
result_dir.mkdir(parents=True, exist_ok=True)
comparisons = []

with torch.inference_mode():
    for image_path in sample_paths:
        original = Image.open(image_path).convert("RGB")
        input_tensor = transform(original).unsqueeze(0).to(device)
        output_tensor = generator(input_tensor)
        generated = tensor_to_pil(
            output_tensor,
            brightness_gain=brightness_gain,
            output_gamma=output_gamma,
        )
        generated = generated.resize(original.size, Image.Resampling.BICUBIC)

        relative_path = image_path.relative_to(INPUT_DIR)
        destination = result_dir / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        generated.save(destination)
        comparisons.append((original, generated, relative_path))

fig, axes = plt.subplots(len(comparisons), 2, figsize=(12, 4 * len(comparisons)))
if len(comparisons) == 1:
    axes = axes.reshape(1, 2)

for row, (original, generated, relative_path) in enumerate(comparisons):
    axes[row, 0].imshow(original)
    axes[row, 0].set_title(f"Input\n{relative_path}")
    axes[row, 1].imshow(generated)
    axes[row, 1].set_title(f"SemGAN output ({DIRECTION}, original size)")
    axes[row, 0].axis("off")
    axes[row, 1].axis("off")

fig.tight_layout()
figure_path = OUTPUT_DIR / f"comparison_{DIRECTION}.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"generated images: {result_dir}")
print(f"comparison figure: {figure_path}")
